In [9]:
import pandas as pd

url = "https://data.bts.gov/resource/keg4-3bc2.csv?$limit=50000"
df = pd.read_csv(url)

print(df.shape)
df.head()

(50000, 10)


,port_name,state,port_code,border,date,measure,value,latitude,longitude,point
0,Richford,Vermont,203,US-Canada Border,2026-01-01T00:00:00.000,Personal Vehicles,4050,45.011740,-72.588559,POINT (-72.588559 45.01174)
1,Naco,Arizona,2603,US-Mexico Border,2026-01-01T00:00:00.000,Trucks,208,31.334084,-109.948413,POINT (-109.948413 31.334084)
2,Naco,Arizona,2603,US-Mexico Border,2026-01-01T00:00:00.000,Truck Containers Empty,103,31.334084,-109.948413,POINT (-109.948413 31.334084)
3,Sumas,Washington,3009,US-Canada Border,2026-01-01T00:00:00.000,Rail Containers Empty,338,49.002388,-122.264805,POINT (-122.264805 49.002388)
4,Vanceboro,Maine,105,US-Canada Border,2026-01-01T00:00:00.000,Rail Containers Loaded,5908,45.568761,-67.428541,POINT (-67.428541 45.568761)


In [10]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 10 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   port_name  50000 non-null  object 
 1   state      49995 non-null  object 
 2   port_code  50000 non-null  int64  
 3   border     50000 non-null  object 
 4   date       50000 non-null  object 
 5   measure    50000 non-null  object 
 6   value      50000 non-null  int64  
 7   latitude   49993 non-null  float64
 8   longitude  49993 non-null  float64
 9   point      49993 non-null  object 
dtypes: float64(2), int64(2), object(6)
memory usage: 3.8+ MB


In [11]:
df.isnull().sum()

port_name    0
state        5
port_code    0
border       0
date         0
measure      0
value        0
latitude     7
longitude    7
point        7
dtype: int64

In [12]:
df.duplicated().sum()

np.int64(0)

In [13]:
df.describe()

,port_code,value,latitude,longitude
count,50000.000000,5.000000e+04,49993.000000,49993.000000
mean,2380.132520,3.466097e+04,43.241964,-98.549539
std,1222.886028,1.330096e+05,8.311686,18.319410
min,103.000000,0.000000e+00,25.951550,-141.001444
25%,2301.000000,1.020000e+02,32.673389,-112.817077
50%,3004.000000,8.115000e+02,46.508611,-100.927612
75%,3323.000000,8.760250e+03,48.999538,-82.423611
max,3814.000000,2.302112e+06,62.614961,-66.980076


In [14]:
print(df["border"].value_counts())
print(df["measure"].value_counts())

border
US-Canada Border    36894
US-Mexico Border    13106
Name: count, dtype: int64
measure
Personal Vehicles              7286
Personal Vehicle Passengers    7283
Trucks                         6600
Truck Containers Empty         6408
Truck Containers Loaded        5906
Pedestrians                    3699
Buses                          2591
Bus Passengers                 2580
Rail Containers Empty          2052
Trains                         2050
Rail Containers Loaded         1985
Train Passengers               1560
Name: count, dtype: int64


In [15]:
# ── 1. Fix date column ──────────────────────────────────────────
df["date"] = pd.to_datetime(df["date"])

# ── 2. Check zero values ────────────────────────────────────────
zero_count = df[df["value"] == 0].shape[0]
print(f"Zero value rows: {zero_count}")

# ── 3. Drop rows missing location data ─────────────────────────
# latitude, longitude, and point are all missing together
# these rows aren't useful for geographic analysis
df.dropna(subset=["latitude", "longitude", "point"], inplace=True)

# ── 4. Fix missing state values ─────────────────────────────────
# Fill missing state using port_name as reference
print(df[df["state"].isnull()][["port_name", "state", "border"]])

# ── 5. Standardize text columns ─────────────────────────────────
df["port_name"] = df["port_name"].str.strip().str.title()
df["state"]     = df["state"].str.strip().str.title()
df["border"]    = df["border"].str.strip()
df["measure"]   = df["measure"].str.strip()

# ── 6. Confirm cleaning worked ──────────────────────────────────
print("\n── After Cleaning ──")
print(f"Shape: {df.shape}")
print(f"Missing values:\n{df.isnull().sum()}")
print(f"Date dtype: {df['date'].dtype}")

Zero value rows: 4
Empty DataFrame
Columns: [port_name, state, border]
Index: []

── After Cleaning ──
Shape: (49993, 10)
Missing values:
port_name    0
state        0
port_code    0
border       0
date         0
measure      0
value        0
latitude     0
longitude    0
point        0
dtype: int64
Date dtype: datetime64[ns]


In [16]:
import os

# Create the folder if it doesn't exist
os.makedirs("data/cleaned", exist_ok=True)

# Save cleaned data
df.to_csv("data/cleaned/cleaned_border_crossing.csv", index=False)

print("Cleaned file saved successfully!")
print(f"Final shape: {df.shape}")

Cleaned file saved successfully!
Final shape: (49993, 10)
